This notebook takes the output of the ortho creation step & precomputes wald

In [23]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster
import pandas as pd
import numpy as np

Set up the cluster

In [3]:
local=False
if local:
    cluster=LocalCluster(memory_limit='48G')
    client = Client(cluster)
else:
    cluster=SLURMCluster(
        cores=8,#cores per slurm job
        memory="128G",#memory per slurm job
        processes=4,#dask workers per slurm job
        job_extra_directives=[# "-p ycga", 
            f"--job-name=simclust_worker",
            f"--time=2:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=8)
    client = Client(cluster,
            timeout=f"{10*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s",  # Worker heartbeat interval,
        )

Let's just do a super simple bounded concurrency approach

In [4]:
from pathlib import Path

In [16]:
# in the real version, de_novo_sim will take a pair, path/name on init and never save it.
# relative paths to individual components can be used, saved, assumed. 
DATA_ROOT=Path("/nfs/roberts/project/pi_skr2/shared/tabula_data")
path=DATA_ROOT/"simulated/shendure_pow_analysis"
name="sim_with_orthos_20251119"

In [17]:
from dask.distributed import Semaphore, as_completed, get_client

In [18]:
ortho_root=path/name/"orthos"
scmpradat_root=path/name/"scMPRA"
# output_root=path/name/"orthos_with_precomputed_wald_erin_numerical_stability_ALL5"
output_root=path/name/"orthos_with_precomputed_wald_erin_numerical_stability_test"
output_root.mkdir(exist_ok=True)

input_ortho_names=[path.name for path in ortho_root.iterdir()]

#Semaphore(max_leases=2, name="wald-precompute")

def precompute_one_wald(input_root, scmpradat_root, name):
    sem = Semaphore(name="wald-precompute")
    with sem:
        client=get_client()
        dat=scm.scMPRA_data.from_parquet(scmpradat_root/Path(name).with_suffix(".scmpra"))
        dat.ortho_filter()
        ortho_oi=scm.ortho.load(client=client,
                                path=input_root,
                                name=name)
        ortho_oi.training_data=dat
        ortho_oi.precompute_wald(client)
        return ortho_oi


        
# futures = [client.submit(precompute_one_wald,) for name_oi in input_ortho_names]

In [19]:
input_ortho_names

['0', '1', '2', '3', '4']

In [20]:
# for one replicate

test_particle=precompute_one_wald(input_root=ortho_root,
        scmpradat_root=scmpradat_root,
        name=input_ortho_names[0])

names=[]
errors=[]
for name in test_particle.wald_precomp.by_cell_type:
    names.append(name)
    errors.append(test_particle.wald_precomp.by_cell_type[name].result().debug_msg)
pd.DataFrame({"name":names,"debug":errors}).to_csv("by_cell_types.tsv",sep="\t")

names=[]
errors=[]
for name in test_particle.wald_precomp.by_cre:
    names.append(name)
    errors.append(test_particle.wald_precomp.by_cre[name].result().debug_msg)
pd.DataFrame({"name":names,"debug":errors}).to_csv("by_cre.tsv",sep="\t")


scMPRAforge: INFO: Dropped 2 of 1010 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [21]:
def inspect_wald_vs_mu(
    ortho_obj,
    ct,
    cre,
    ref_cre="reference",
):
    """
    Debug helper to compare:
      - beta used in Wald (log fold-change coefficient)
      - exp(beta) (Wald fold_change)
      - 'true' mu ratio from by_cell_type_parameters (on data scale)

    Parameters
    ----------
    ortho_obj : scm.ortho
        Your loaded ortho object (e.g. test_particle).
    ct : str
        Comparison cell type (e.g. "Mesoderm").
    cre : str
        Comparison CRE (e.g. "Cited2_chr10_1248").
    ref_cre : str, default "reference"
        Name of the reference CRE in the design matrix / params.
    """

    # ---- 1. Build the same bundle the Wald code uses ----
    bundle = ortho_obj.make_wald_eval_bundle()
    blk = bundle["by_cell_type"].get(str(ct))
    if blk is None:
        raise ValueError(f"No by_cell_type block for cell_type={ct!r}")

    xmu_names = blk["xmu_names"]
    xmu = np.asarray(blk["xmu"])

    # sanity check lengths
    print(f"[by_cell_type] cell_type={ct}")
    print(f"  # of x_mu coefficients: {len(xmu_names)}")

    # ---- 2. Find the column for this CRE (same logic as wald) ----
    # This uses the exact helper the real code calls.
    from scMPRAforge import find_treatment_column  # adjust import path if needed

    col_name = find_treatment_column(xmu_names, "cre_id", cre)
    if col_name is None:
        raise ValueError(f"No treatment column found for cre={cre!r} in cell_type={ct!r}")

    j = xmu_names.index(col_name)
    beta = float(xmu[j])
    fc_wald = float(np.exp(beta))

    print("\n[WALD beta / FC]")
    print(f"  column name      : {col_name}")
    print(f"  beta (log-FC)    : {beta:.6g}")
    print(f"  exp(beta) (FC)   : {fc_wald:.6g}")

    # ---- 3. Get 'true' mu on data scale from by_cell_type_parameters ----
    if ortho_obj.by_cell_type_parameters is None:
        raise RuntimeError("by_cell_type_parameters is None; run ortho.extract_params(...) first.")

    params_ct = ortho_obj.by_cell_type_parameters.flattened_copy()  # resolve futures
    mu_df = params_ct.nb[ct]  # DataFrame indexed by CRE names, column 'mu'

    if "mu" not in mu_df.columns:
        raise ValueError("Expected 'mu' column in by_cell_type_parameters.nb[ct]")

    if cre not in mu_df.index:
        raise ValueError(f"CRE {cre!r} not found in mu_df index for ct={ct!r}")
    if ref_cre not in mu_df.index:
        raise ValueError(
            f"Reference CRE {ref_cre!r} not found in mu_df index for ct={ct!r}.\n"
            f"Available CREs (head): {mu_df.index[:10].tolist()}"
        )

    mu_comp = float(mu_df.loc[cre, "mu"])
    mu_ref  = float(mu_df.loc[ref_cre, "mu"])

    fc_mu = mu_comp / mu_ref if mu_ref > 0 else np.nan
    lfc_mu = np.log(mu_comp) - np.log(mu_ref) if (mu_comp > 0 and mu_ref > 0) else np.nan

    print("\n[Parameter-derived mu]")
    print(f"  mu_comp (data scale) : {mu_comp:.6g}")
    print(f"  mu_ref  (data scale) : {mu_ref:.6g}")
    print(f"  mu_comp / mu_ref     : {fc_mu:.6g}")
    print(f"  log(mu_comp) - log(mu_ref) : {lfc_mu:.6g}")

    # ---- 4. Compare ----
    print("\n[Comparison]")
    print(f"  |beta - log(mu_comp/mu_ref)| : {abs(beta - lfc_mu):.6g}")
    print(f"  Wald FC vs mu-ratio         : FC_wald={fc_wald:.6g}  FC_from_mu={fc_mu:.6g}")

    return {
        "beta": beta,
        "fc_wald": fc_wald,
        "mu_comp": mu_comp,
        "mu_ref": mu_ref,
        "fc_mu": fc_mu,
        "lfc_mu": lfc_mu,
    }

In [24]:
# pick a specific simulated CT / CRE to inspect
ct_debug  = "EpiblastPrimitiveStreak"           # or whatever cell type you're worried about
cre_debug = "inactive_0"  # or another CRE Mackenzie flagged
ref_cre   = "reference"          # adjust if your design uses a different label

debug_res = inspect_wald_vs_mu(test_particle, ct=ct_debug, cre=cre_debug, ref_cre=ref_cre)

[by_cell_type] cell_type=EpiblastPrimitiveStreak
  # of x_mu coefficients: 101

[WALD beta / FC]
  column name      : C(cre_id, contr.treatment(base='reference'))[T.inactive_0]
  beta (log-FC)    : -3.28044
  exp(beta) (FC)   : 0.0376116

[Parameter-derived mu]
  mu_comp (data scale) : 0.0261937
  mu_ref  (data scale) : 0.696428
  mu_comp / mu_ref     : 0.0376116
  log(mu_comp) - log(mu_ref) : -3.28044

[Comparison]
  |beta - log(mu_comp/mu_ref)| : 4.44089e-16
  Wald FC vs mu-ratio         : FC_wald=0.0376116  FC_from_mu=0.0376116


In [8]:
# for all reps
for i in input_ortho_names:
    test_particle=precompute_one_wald(input_root=ortho_root,
        scmpradat_root=scmpradat_root,
        name=i)

    names=[]
    errors=[]
    for name in test_particle.wald_precomp.by_cell_type:
        names.append(name)
        errors.append(test_particle.wald_precomp.by_cell_type[name].result().debug_msg)
    pd.DataFrame({"name":names,"debug":errors}).to_csv(f"{i}_by_cell_types.tsv",sep="\t")


    names=[]
    errors=[]
    for name in test_particle.wald_precomp.by_cre:
        names.append(name)
        errors.append(test_particle.wald_precomp.by_cre[name].result().debug_msg)
    pd.DataFrame({"name":names,"debug":errors}).to_csv(f"{i}_by_cre.tsv",sep="\t")

    del test_particle



scMPRAforge: INFO: Dropped 618 of 2080 (cell_type, cre_id) combos with fewer than 3 nonzero entries.
scMPRAforge: INFO: Dropped 619 of 2080 (cell_type, cre_id) combos with fewer than 3 nonzero entries.
scMPRAforge: INFO: Dropped 618 of 2080 (cell_type, cre_id) combos with fewer than 3 nonzero entries.
scMPRAforge: INFO: Dropped 618 of 2080 (cell_type, cre_id) combos with fewer than 3 nonzero entries.
scMPRAforge: INFO: Dropped 618 of 2080 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [19]:
client.close()
cluster.close()

In [22]:
!rm ./worker*
!rm *by_cell_types.tsv
!rm *by_cre.tsv

rm: cannot remove './worker*': No such file or directory


In [21]:
!rm -r $output_root